In [162]:
import os
import fitz
import sys
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any, Tuple, Dict
from abc import abstractmethod, ABC
import random
import numpy as np
import pandas as pd
import copy
import nltk
from nltk.corpus import stopwords
import pickle
import itertools
from dataclasses import dataclass, asdict
from tqdm import tqdm
from enum import Enum
from dotenv import load_dotenv

sys.path.append("../")

load_dotenv(dotenv_path="../.env")
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [163]:
from utils import file_handle, grouping
from utils.tree import Node

In [164]:
from dataset_utils.reader import ADIQDataset
from dataset_utils.question import Question


ds = ADIQDataset("../dataset/datasets/simpleV3.1")

In [165]:
rule_information = file_handle.load_pickle(
    os.path.join("../dataset/test_datasets/TEST_TreeStruct.pkl")
)

asset_descriptions = ds.asset_descriptions

# Observation stats 

In [166]:
total_obs = list()
unq_obs = set()
for d in rule_information["rule_set"]:
    unq_obs = unq_obs.union(set(d["display_text"]["observations"]))
    total_obs.extend(d["display_text"]["observations"])

len(total_obs), len(unq_obs)

unq_obs = list(unq_obs)

In [167]:
for i, unq in enumerate(unq_obs):
    print("{} : {}".format(i,unq))

0 : Boiler tube fouling
1 : Economizer fouling
2 : Condenser water high temperature
3 : Chiller refrigerant leak
4 : Flow meter calibration
5 : Blowdown valve failure
6 : Outside air temperature sensor failure
7 : Cooling tower reset in manual
8 : Condenser water low flow
9 : Increased load on PDU
10 : Temperature sensor calibration
11 : Too few chillers running
12 : Excessive load
13 : Dryer not functioning
14 : Fuel supply valve issue
15 : Too many pumps running
16 : Make up water valve issue
17 : Chilled water valve failure
18 : Power Meter Current Transformer issue
19 : System leak
20 : Conductivity meter issue causing excessive blowdown
21 : Chilled water valve maunal override
22 : Pressure sensor calibration
23 : Air to air heat exhanger clogged/dirty
24 : Too few pumps running
25 : Chilled water high return temp
26 : Chilled water flow meter calibration
27 : Level sensor issue
28 : VFD speed control in manual
29 : Chilled water high flow
30 : Water to aire heat exhanger fouled
3

In [168]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [169]:
enc = model.encode(unq_obs)

In [170]:
norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

In [171]:
_thresh = 0.992
cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

thresh = np.quantile(cosine_sim.flatten(),_thresh)
print(thresh)
for i in range(cosine_sim.shape[0]):
    for j in range(i, cosine_sim.shape[0]):
        cosine_sim[i,j] = 0

ind = np.where(cosine_sim>thresh)

observation_map = {}

edges = []
for x,y in zip(ind[0], ind[1]):
    edges.append((x,y))
    
res = grouping.getComponents(cosine_sim.shape[0], edges)


0.99999917


In [172]:
obs_mapping = {}
for r in res:
    if len(r)>1:
        major = np.argmax([len(unq_obs[x]) for x in r])
        major = unq_obs[r[major]]
        
        for r0 in r:
            obs_mapping[unq_obs[r0]] = major 
    else:
        obs_mapping[unq_obs[r[0]]] = unq_obs[r[0]]

# Rule stats 

In [173]:
rule_information = file_handle.load_json(
    os.path.join("../dataset/test_datasets/TEST_rule_information.json")
)

In [174]:
total_rules = list()
unq_rules = set()
for d in rule_information["rule_set"]: 
    s = (d["display_text"]["rules"].replace("-","").replace("OR", "").replace("(", "").replace(")", "")).split("\n")
    s = [x.strip() for x in s]
    unq_rules = unq_rules.union(set(s))
    total_rules.extend(s)

unq_rules = list(unq_rules)

In [175]:
len(total_rules), len(unq_rules)

(47, 38)

In [176]:
unq_rules

['Cooling Tower Running',
 "Sum of Child Pumps' KW / Condenser Water Flow > xx",
 'Chilled water Supply temp  Chilled water supply temp setpoint  < 2 degF',
 'Condenser Water Flow > 50 gallon/minute',
 'ABSSteam Pressure  135 PSI > 15 PSIG IF Steam Pressure Setpoint NOT Reporting',
 'VFD Speed % > 97%',
 'Supply Temp Setpoint = Previous Hour Supply Air Temp Setpoint',
 'Coolant Flow > 50 g/m',
 'Power Input > 5 Standard Deviations from past 7 day average',
 'Met for 3 Hours Checking Previous 3 Days Daily Average',
 'Power Input > 0.1KW',
 'Number of Chillers Running is > 1',
 'Chiller Running',
 'Chilled Water Supply Temp  Chilled Water Supply Temp Setpoint  > 4 degF',
 'Hot Water Boiler Gallons/BTU efficiency > XXX',
 'Boiler Running',
 'Fire Rate > 20%',
 'Chiller % Loaded> 97%',
 '55 degF < Outside Air Temp < 80 degF',
 'ABSSteam Pressure  Steam Pressure Setpoint > 15 PSIG',
 'Discharge Air Temperature > 75 degF',
 'Met for 2 Hours',
 'Air Compressor Running',
 'Met for 6 hours',
 '

In [177]:
enc = model.encode(unq_rules)

In [178]:
norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

In [179]:
_thresh = 0.997
cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

thresh = np.quantile(cosine_sim.flatten(),_thresh)
print(thresh)
for i in range(cosine_sim.shape[0]):
    for j in range(i, cosine_sim.shape[0]):
        cosine_sim[i,j] = 0

ind = np.where(cosine_sim>thresh)

edges = []
for x,y in zip(ind[0], ind[1]):
    edges.append((x,y))
    
res = grouping.getComponents(cosine_sim.shape[0], edges)


0.9999993


In [180]:
rules_mapping = {}
for r in res:
    if len(r)>1:
        major = np.argmax([len(unq_rules[x]) for x in r])
        major = unq_rules[r[major]]

        print("Major: \n{}".format(major))
        
        for r0 in r:
            print(unq_rules[r0])
            rules_mapping[unq_rules[r0]] = major 

        print("\n")
    else:
        rules_mapping[unq_rules[r[0]]] = unq_rules[r[0]]

# Rule Grouping with semantic understanding

In [181]:
from sentence_transformers import SentenceTransformer, util


model = SentenceTransformer("all-mpnet-base-v2")

In [182]:
rule_repr_dict:Dict[str,str] = {x["id"]:x["display_text"]["rules"] for x in rule_information["rule_set"]}
enc = model.encode(list(rule_repr_dict.values()))
rule_index_map = {k:v for v,k in enumerate(rule_repr_dict.keys())}
index_rule_map = {k:v for v,k in rule_index_map.items()}

norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

In [183]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(cosine_sim)

plt.savefig('RRSim_heatmap.png', dpi=600, bbox_inches='tight')

<Figure size 640x480 with 2 Axes>

In [184]:
rule_codition_similarity = np.argsort(cosine_sim, axis=-1)

rule_id_similarity_map = {}
for k,v in rule_index_map.items():
    rule_id_similarity_map[k] = [index_rule_map[x] for x in rule_codition_similarity[v]]

# Dataset Creation 

In [185]:
random.seed(420)
np.random.seed(420)

In [186]:
rule_information = file_handle.load_pickle(
    os.path.join("../dataset/test_datasets/TEST_TreeStruct.pkl")
)

asset_descriptions = ds.asset_descriptions

In [187]:
for  x in rule_information["rule_set"]:
    if x["id"] == "TEST_RULE_CT00004":
        print(x["display_text"]["rules"])

In [188]:
from dataset_utils import creation, config


dataset_config = config.DatasetCreationConfig({
    "num_options":4,
    "question_template_mode":config.QuestionTemplateMode.random,
    "sel_random_sample": 2,
    'elem_random_sample':2,
    "question_template_random_count" : 1,
    "n_least_sim_rules":2
})

In [189]:
dataset_config.to_config()

{'subject': 'diagnosis_sense_simple_question_analysis',
 'db_acronym': 'ADIQ',
 'db_name': 'AssetDiagnosisIQ',
 'option_choices_string': 'ABCDEFGHIJKLMNOPQRSTUVWXYZ',
 'question_first': None,
 'text_type': 'choice',
 'relevancy': None,
 'num_options': 4,
 'question_template_mode': 'random',
 'question_template_pre_select_options': [0],
 'question_template_random_count': 1,
 'sel_random_sample': 2,
 'elem_random_sample': 2,
 'n_least_sim_rules': 2,
 'question_condition_template': '\n## Asset Description:\n{asset_type}: {asset_description}\n\n## Conditions:\n{conditions}\n\n{question_prompt}\n'}

In [190]:
dataset = creation.create_dataset(
    rule_information,
    obs_mapping,
    asset_descriptions,
    rule_id_similarity_map,
    dataset_config
    )

In [191]:
len(dataset["questions"])

176

In [192]:
creation.save_dataset(
    dataset,
    "simpleTestV1",
    "test_datasets/datasets"
)

Overridding Dataset


In [193]:
from dataset_utils.reader import ADIQDataset
from dataset_utils.outputs import to_basic_prompt

ds = ADIQDataset("test_datasets/datasets/simpleTestV1")

In [194]:
len(ds.questions)

176

In [195]:
from collections import Counter

for que1 in tqdm(ds.questions):
    flag = True
    dup_id = None
    for que2 in ds.questions:
        if que1.id == que2.id:
            continue

        if que1.asset_type == que2.asset_type:
            if que1.condition_description == que2.condition_description:
                if Counter(que1.options) == Counter(que2.options):
                    print(que1, que2)
                    flag = False
                    dup_id = que2.id

    if not flag:
        print(que1.id, dup_id)

100%|██████████| 176/176 [00:00<00:00, 11411.31it/s]


In [196]:
question_template = """
## Asset Description:
{asset_type}: {asset_description}

## Conditions:
{conditions}

## How long the conditions were met:
{temporal_condition}

{question_prompt}
{options}
"""

In [197]:
print(to_basic_prompt(ds.questions[100],question_template, ds.asset_descriptions))


## Asset Description:
Chiller: A device that removes heat from a liquid through a vapor-compression or vapor-absorption refrigeration process. The cooled liquid can then be used to cool equipment, buildings, or other systems.

## Conditions:
- Chiller Running
- VFD Speed % > 97%
- (Chilled Water Supply Temp - Chilled Water Supply Temp Setpoint ) < -2 degF

## How long the conditions were met:
Met for 4 Hours

Given what is known about the asset’s conditions, which explanation makes the MOST sense?
A). Too few pumps running
B). System leak
C). Power Meter Current Transformer issue
D). Conductivity meter issue causing excessive blowdown



In [198]:
ds.questions[100].rule_id, ds.questions[100].correct

(5, [True, False, False, False])

In [199]:
q_type_stats = {"p":0,"n":0}
for q in ds.questions:
    if q.question_type == "positive":
        q_type_stats["p"] += 1
    else:
        q_type_stats["n"] += 1

In [200]:
q_type_stats

{'p': 142, 'n': 34}

In [201]:
opt_comp = {}
for q in ds.questions:
    try:
        opt_comp[len(q.options)] += 1
    except KeyError as er:
        opt_comp[len(q.options)] = 1

In [202]:
opt_comp

{4: 142, 3: 34}